# LangSmith 可观测性

trace 是你的应用从输入到输出所经历的一系列步骤。其中每一个单独的步骤都由一个 run 表示。你可以使用 [LangSmith](https://smith.langchain.com) 来可视化这些执行步骤。这可以让你做到以下几点:

* [调试本地运行的应用](https://docs.langchain.com/langsmith/observability-studio#debug-langsmith-traces)。
* [评估应用性能](https://docs.langchain.com/oss/langchain/test/evals)。
* [监控应用](https://docs.langchain.com/langsmith/dashboards)。

## 选择性追踪

你可以选择使用 LangSmith 的 `tracing_context` 上下文管理器, 只追踪应用中的特定调用或部分:

```python
import langsmith as ls

# This WILL be traced
with ls.tracing_context(enabled=True):
    agent.invoke({"messages": [{"role": "user", "content": "Send a test email to alice@example.com"}]})

# This will NOT be traced (if LANGSMITH_TRACING is not set)
agent.invoke({"messages": [{"role": "user", "content": "Send another email"}]})
```

## 记录到项目

### 静态设置

你可以通过设置 `LANGSMITH_PROJECT` 环境变量, 为整个应用设置自定义项目名:

```bash
export LANGSMITH_PROJECT=my-agent-project
```

### 动态设置

你可以通过编程方式为特定操作设置项目名:

```python
import langsmith as ls

with ls.tracing_context(project_name="email-agent-test", enabled=True):
    response = agent.invoke({
        "messages": [{"role": "user", "content": "Send a welcome email"}]
    })
```

## 为 trace 添加 metadata

你可以用自定义的 metadata 与 tags 来标注 trace:

```python
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Send a welcome email"}]},
    config={
        "tags": ["production", "email-assistant", "v1.0"],
        "metadata": {
            "user_id": "user_123",
            "session_id": "session_456",
            "environment": "production"
        }
    }
)
```

`tracing_context` 也接受 tags 与 metadata, 以实现更细粒度的控制:

```python
with ls.tracing_context(
    project_name="email-agent-test",
    enabled=True,
    tags=["production", "email-assistant", "v1.0"],
    metadata={"user_id": "user_123", "session_id": "session_456", "environment": "production"}):
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "Send a welcome email"}]}
    )
```

这些自定义的 metadata 与 tags 会被附加到 LangSmith 中的 trace 上。

> **提示**
>
> 要详细了解如何使用 trace 来调试、评估和监控你的 agent, 请参阅 [LangSmith 文档](https://docs.langchain.com/langsmith/observability)。

## 使用 anonymizer 防止在 trace 中记录敏感数据

你可能希望遮蔽敏感数据, 以防其被记录到 LangSmith。你可以创建 [anonymizer](https://docs.langchain.com/langsmith/mask-inputs-outputs#rule-based-masking-of-inputs-and-outputs), 并通过配置将它们应用到你的 graph 上。本示例会从发送到 LangSmith 的 trace 中脱敏掉所有匹配社会安全号码 (Social Security Number) 格式 XXX-XX-XXXX 的内容。

```python
from langchain_core.tracers.langchain import LangChainTracer
from langgraph.graph import StateGraph, MessagesState
from langsmith import Client
from langsmith.anonymizer import create_anonymizer

anonymizer = create_anonymizer([
    # Matches SSNs
    { "pattern": r"\b\d{3}-?\d{2}-?\d{4}\b", "replace": "<ssn>" }
])

tracer_client = Client(anonymizer=anonymizer)
tracer = LangChainTracer(client=tracer_client)
# Define the graph
graph = (
    StateGraph(MessagesState)
    ...
    .compile()
    .with_config({'callbacks': [tracer]})
)
```